In [1]:
# =========================================================
# AutoEncoder - 이상 탐지 실험
# 데이터셋: NSL-KDD, UNSW-NB15
# 전처리  : MinMax / Quantile
# 임계값  : Bootstrap (α=0.10, α=0.15)
# 범주형  : One-Hot Encoding (train 기준 fit)
# =========================================================

import numpy as np
import pandas as pd
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, confusion_matrix
)

# =========================================================
# 경로 설정
# =========================================================
DATA_DIR = "./data/"

NSL_CAT  = ["protocol_type", "service", "flag"]
UNSW_CAT = ["service", "proto", "state"]

# =========================================================
# Config
# =========================================================
CFG = {
    "seed":         42,
    "hidden_dim_1": 32,
    "hidden_dim_2": 24,
    "latent_dim":   16,
    "lr":           0.001,
    "batch_size":   128,
    "max_epochs":   500,
    "patience":     20,
    "min_delta":    1e-6,
    "B":            500,
}

# =========================================================
# Seed & Device
# =========================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# AutoEncoder
# =========================================================
class AutoEncoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, CFG["hidden_dim_1"]), nn.Tanh(),
            nn.Linear(CFG["hidden_dim_1"], CFG["hidden_dim_2"]), nn.Tanh(),
            nn.Linear(CFG["hidden_dim_2"], CFG["latent_dim"]), nn.Tanh(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(CFG["latent_dim"], CFG["hidden_dim_2"]), nn.Tanh(),
            nn.Linear(CFG["hidden_dim_2"], CFG["hidden_dim_1"]), nn.Tanh(),
            nn.Linear(CFG["hidden_dim_1"], input_dim), nn.Tanh(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

# =========================================================
# 재구성 오차
# =========================================================
@torch.no_grad()
def reconstruction_error(x_tensor, model):
    model.eval()
    errs = []
    for i in range(0, len(x_tensor), CFG["batch_size"]):
        xb  = x_tensor[i:i + CFG["batch_size"]].to(device)
        err = torch.mean((xb - model(xb)) ** 2, dim=1)
        errs.append(err.cpu())
    return torch.cat(errs).numpy()

# =========================================================
# 학습 (Early Stopping)
# =========================================================
def train_ae(X_train, X_val):
    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train, dtype=torch.float32)),
        batch_size=CFG["batch_size"], shuffle=True
    )
    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
    model   = AutoEncoder(X_train.shape[1]).to(device)
    opt     = optim.Adam(model.parameters(), lr=CFG["lr"])
    loss_fn = nn.MSELoss()

    best_val_loss      = np.inf
    patience_counter   = 0
    best_state         = None

    for epoch in range(CFG["max_epochs"]):
        model.train()
        for (xb,) in train_loader:
            xb = xb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), xb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), X_val_t).item()

        if val_loss < best_val_loss - CFG["min_delta"]:
            best_val_loss    = val_loss
            best_state       = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= CFG["patience"]:
                break

    if best_state:
        model.load_state_dict(best_state)
    return model

# =========================================================
# OHE 인코딩 (train 기준 fit)
# =========================================================
def encode_df(df, cat_cols, label_col, ohe_columns=None):
    if label_col == "class":
        y = (df[label_col] != "normal").astype(int).values
    else:
        y = df[label_col].values.astype(int)

    ohe = pd.get_dummies(df[cat_cols].fillna("NaN"), drop_first=False)
    if ohe_columns is None:
        ohe_columns = ohe.columns.tolist()
    else:
        ohe = ohe.reindex(columns=ohe_columns, fill_value=0)

    num_cols = [c for c in df.columns if c != label_col and c not in cat_cols]
    X = np.concatenate([
        df[num_cols].values.astype(np.float32),
        ohe.values.astype(np.float32)
    ], axis=1)
    return X, y, ohe_columns

# =========================================================
# Bootstrap 임계값
# =========================================================
def bootstrap_threshold(scores, percentiles=(90, 85), B=500, seed=42):
    rng     = np.random.default_rng(seed)
    results = []
    for _ in range(B):
        sample = rng.choice(scores, size=len(scores), replace=True)
        results.append(np.percentile(sample, percentiles))
    return np.median(np.array(results), axis=0)  # (th_p90, th_p85)

# =========================================================
# 성능 평가
# =========================================================
def evaluate(test_scores, y_true, thresholds_dict):
    try:
        auc = roc_auc_score(y_true, test_scores)
    except ValueError:
        auc = np.nan

    rows = []
    for alpha, T in thresholds_dict.items():
        y_pred = (test_scores >= T).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        rows.append({
            "α":           alpha,
            "Precision":   precision_score(y_true, y_pred, zero_division=0),
            "Recall":      recall_score(y_true, y_pred, zero_division=0),
            "Specificity": tn / (tn + fp) if (tn + fp) else 0.0,
            "F1-score":    f1_score(y_true, y_pred, zero_division=0),
            "Accuracy":    accuracy_score(y_true, y_pred),
            "AUC":         auc,
        })
    return pd.DataFrame(rows).sort_values("α").reset_index(drop=True)

# =========================================================
# 파이프라인
# =========================================================
def run_pipeline(train_path, valid_path, test_path,
                 cat_cols, label_col):

    set_seed(CFG["seed"])  # 실험마다 seed 리셋

    df_train = pd.read_csv(train_path)
    df_valid = pd.read_csv(valid_path)
    df_test  = pd.read_csv(test_path)

    # OHE: train 기준 fit, valid/test는 transform
    X_train, _,      ohe_cols = encode_df(df_train, cat_cols, label_col)
    X_valid, _,      _        = encode_df(df_valid, cat_cols, label_col, ohe_columns=ohe_cols)
    X_test,  y_true, _        = encode_df(df_test,  cat_cols, label_col, ohe_columns=ohe_cols)

    model = train_ae(X_train, X_valid)

    # valid 재구성 오차로 Bootstrap 임계값 산출
    val_scores      = reconstruction_error(torch.tensor(X_valid, dtype=torch.float32), model)
    th_p90, th_p85  = bootstrap_threshold(val_scores, percentiles=(90, 85))

    # α=0.10 → P90, α=0.15 → P85
    thresholds = {0.10: float(th_p90), 0.15: float(th_p85)}

    test_scores = reconstruction_error(torch.tensor(X_test, dtype=torch.float32), model)
    return evaluate(test_scores, y_true, thresholds)

# =========================================================
# 결과 출력
# =========================================================
def print_results(dataset_name, df_mm, df_qt):
    COLS    = ["Precision", "Recall", "Specificity", "F1-score", "Accuracy", "AUC"]
    COLS_KR = ["정밀도",    "민감도",  "특이도",      "F1 점수",  "정확도",   "AUC"]
    W       = 78

    print("=" * W)
    print(f" {dataset_name}")
    print("=" * W)
    print(f"  {'':22s}" + "".join(f"{k:>8}" for k in COLS_KR))
    print("-" * W)

    for scaler, df in [("MinMax", df_mm), ("Quantile", df_qt)]:
        for _, row in df.iterrows():
            label = f"  {scaler:<10} α={row['α']:.2f}"
            vals  = "".join(f"{row[c]:>8.2f}" for c in COLS)
            print(f"{label:<26}{vals}")
        print("-" * W)
    print()

# =========================================================
# 실험 실행
# =========================================================
if __name__ == "__main__":

    print(f"Using device: {device}\n")

    # ── NSL-KDD ──────────────────────────────────────────
    res_nsl_mm = run_pipeline(
        DATA_DIR + "NSL_KDD_MinMax_train_normal_80.csv",
        DATA_DIR + "NSL_KDD_MinMax_train_normal_20.csv",
        DATA_DIR + "NSL_KDD_MinMax_test.csv",
        cat_cols=NSL_CAT, label_col="class"
    )
    res_nsl_qt = run_pipeline(
        DATA_DIR + "NSL_KDD_Quantile_train_normal_80.csv",
        DATA_DIR + "NSL_KDD_Quantile_train_normal_20.csv",
        DATA_DIR + "NSL_KDD_Quantile_test.csv",
        cat_cols=NSL_CAT, label_col="class"
    )

    # ── UNSW-NB15 ────────────────────────────────────────
    res_unsw_mm = run_pipeline(
        DATA_DIR + "UNSW_NB15_MinMax_train_normal_80.csv",
        DATA_DIR + "UNSW_NB15_MinMax_train_normal_20.csv",
        DATA_DIR + "UNSW_NB15_MinMax_test.csv",
        cat_cols=UNSW_CAT, label_col="label"
    )
    res_unsw_qt = run_pipeline(
        DATA_DIR + "UNSW_NB15_Quantile_train_normal_80.csv",
        DATA_DIR + "UNSW_NB15_Quantile_train_normal_20.csv",
        DATA_DIR + "UNSW_NB15_Quantile_test.csv",
        cat_cols=UNSW_CAT, label_col="label"
    )

    # ── 출력 ─────────────────────────────────────────────
    print_results("NSL-KDD",   res_nsl_mm,  res_nsl_qt)
    print_results("UNSW-NB15", res_unsw_mm, res_unsw_qt)

Using device: cpu

 NSL-KDD
                             정밀도     민감도     특이도   F1 점수     정확도     AUC
------------------------------------------------------------------------------
  MinMax     α=0.10           0.91    0.87    0.88    0.89    0.87    0.94
  MinMax     α=0.15           0.90    0.90    0.86    0.90    0.88    0.94
------------------------------------------------------------------------------
  Quantile   α=0.10           0.90    0.91    0.86    0.90    0.89    0.97
  Quantile   α=0.15           0.88    0.96    0.82    0.91    0.90    0.97
------------------------------------------------------------------------------

 UNSW-NB15
                             정밀도     민감도     특이도   F1 점수     정확도     AUC
------------------------------------------------------------------------------
  MinMax     α=0.10           0.86    0.76    0.85    0.81    0.80    0.88
  MinMax     α=0.15           0.81    0.81    0.77    0.81    0.79    0.88
------------------------------------------------